# CSE 151B Competition — Starter Notebook (Kaggle T4×2)

End-to-end pipeline:

1. Install packages into the Kaggle runtime
2. Load the competition dataset
3. Run inference with **Qwen3-4B-Thinking** via vLLM on both T4 GPUs — saves responses to disk immediately
4. Write `submission.csv` **before** scoring (scoring bugs can't kill the submission)
5. Score public responses locally for accuracy feedback

> **Kaggle setup — before running:**
> 1. Accelerator → **GPU T4 x2** (right panel → Session options)
> 2. Add competition data: right panel → Add data → search `cse-151-b-spring-2026-competition`
> 3. Add your utils dataset (contains `judger.py` + `utils.py`). The scoring cell **auto-detects** it via glob, so whatever slug Kaggle assigned is fine.


## 1. Environment Setup

Install required packages into the Kaggle runtime and fix the libcuda symlink so vLLM can link against the NVIDIA driver.

> **After running this cell, restart the session** (Run → Restart session) so the newly installed packages are picked up.


### Comment Out the cell below after first installation.

In [ ]:
!pip install -q vllm transformers tqdm "antlr4-python3-runtime==4.11.1" "protobuf<6.0"

import subprocess, os

# Kaggle T4: libcuda.so.1 lives in /usr/local/nvidia/lib64 which ldconfig doesn't index.
# Check known paths directly, then fall back to ldconfig.
KNOWN_PATHS = [
    "/usr/local/nvidia/lib64/libcuda.so.1",
    "/usr/lib/x86_64-linux-gnu/libcuda.so.1",
    "/usr/lib/libcuda.so.1",
]
out = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout
ldconfig_cands = [l.split("=>")[1].strip() for l in out.splitlines() if "libcuda.so.1" in l]
all_cands = KNOWN_PATHS + ldconfig_cands
libcuda = next((p for p in all_cands if os.path.exists(p) and "stubs" not in p), None) \
       or next((p for p in all_cands if os.path.exists(p)), None)

if not libcuda:
    # Last resort: find it anywhere on the filesystem
    result = subprocess.run(["find", "/usr", "-name", "libcuda.so.1", "-not", "-path", "*/stubs/*"],
                            capture_output=True, text=True)
    found = [p.strip() for p in result.stdout.splitlines() if p.strip()]
    libcuda = found[0] if found else None

assert libcuda, "libcuda.so.1 not found anywhere — is this a GPU runtime?"
print(f"Real libcuda: {libcuda}")

stub_dir = "/kaggle/working/cuda_stubs"
os.makedirs(stub_dir, exist_ok=True)
stub = f"{stub_dir}/libcuda.so"
if os.path.lexists(stub): os.remove(stub)
os.symlink(libcuda, stub)
for var in ("LIBRARY_PATH", "LD_LIBRARY_PATH"):
    os.environ[var] = f"{stub_dir}:{os.environ.get(var, '')}"
subprocess.run(["rm", "-rf", "/root/.cache/flashinfer"], check=False)

print("Install complete. → Run → Restart session, then run the next cell.")


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [ ]:
import json
import importlib.util
import os
import re
import sys
import glob
from pathlib import Path
from typing import Optional

# ─── Must be set BEFORE importing vllm ───────────────────────────────────────
os.environ["VLLM_USE_V1"]                    = "0"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["PYTORCH_ALLOC_CONF"]             = "expandable_segments:True"

# Re-expose libcuda stub dir in case of kernel restart
_stub_dir = "/kaggle/working/cuda_stubs"
if os.path.isdir(_stub_dir):
    for _var in ("LIBRARY_PATH", "LD_LIBRARY_PATH"):
        os.environ[_var] = f"{_stub_dir}:{os.environ.get(_var, '')}"

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

# ── Experiment selection ─────────────────────────────────────────────────────
EXPERIMENT = "exp_006_baseline_replay"

def _find_exp_file(exp_name, filename):
    """Search local path then any attached /kaggle/input dataset."""
    candidates = [
        f"experiments/{exp_name}/{filename}",
        *glob.glob(f"/kaggle/input/**/experiments/{exp_name}/{filename}", recursive=True),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return None

_cfg_path = _find_exp_file(EXPERIMENT, "config.json")
if _cfg_path:
    with open(_cfg_path) as _f:
        EXP_CONFIG = json.load(_f)
    print(f"Loaded config from {_cfg_path}")
else:
    EXP_CONFIG = {}
    print(f"WARNING: No config.json found for {EXPERIMENT!r} — using notebook defaults")

# ── Paths ────────────────────────────────────────────────────────────────────
MODEL_ID          = EXP_CONFIG.get("model_id", "Qwen/Qwen3-4B-Thinking-2507")
PRIVATE_DATA_PATH = "/kaggle/input/competitions/cse-151-b-spring-2026-competition/private.jsonl"
SAMPLE_SUB_PATH   = "/kaggle/input/competitions/cse-151-b-spring-2026-competition/sample_submission.csv"

WORKING_DIR       = Path("/kaggle/working")
PUBLIC_RESP_PATH  = WORKING_DIR / "public_responses.jsonl"
PRIVATE_RESP_PATH = WORKING_DIR / "private_responses.jsonl"
SUBMISSION_PATH   = WORKING_DIR / "submission.csv"
RESULTS_PATH      = WORKING_DIR / "results" / f"{EXPERIMENT}_results.jsonl"

# Auto-detect judger.py across every attached /kaggle/input dataset.
_judger_hits = glob.glob("/kaggle/input/**/judger.py", recursive=True)
JUDGER_DIR = os.path.dirname(_judger_hits[0]) if _judger_hits else None
print(f"Judger dir: {JUDGER_DIR}  (None → scoring will be skipped, submission still OK)")

# ── Inference config (from experiment config.json) ───────────────────────────
MAX_TOKENS   = EXP_CONFIG.get("max_tokens", 8192)
TEMPERATURE  = EXP_CONFIG.get("temperature", 0.6)
TOP_P        = EXP_CONFIG.get("top_p", 0.95)
TOP_K        = EXP_CONFIG.get("top_k", 20)
NUM_SAMPLES  = EXP_CONFIG.get("num_samples", 1)
SPLIT        = EXP_CONFIG.get("split", "all")

# ── Data path — respect split setting ────────────────────────────────────────
_full_public = "/kaggle/input/competitions/cse-151-b-spring-2026-competition/public.jsonl"
if SPLIT == "dev":
    _dev_candidates = [
        "data/splits/dev.jsonl",
        *glob.glob("/kaggle/input/**/data/splits/dev.jsonl", recursive=True),
    ]
    _dev_path = next((c for c in _dev_candidates if os.path.exists(c)), None)
    DATA_PATH = _dev_path if _dev_path else _full_public
    if not _dev_path:
        print("WARNING: dev.jsonl not found, falling back to full public set")
else:
    DATA_PATH = _full_public
print(f"Data path  : {DATA_PATH}")

# ── vLLM engine config ────────────────────────────────────────────────────────
_vllm = EXP_CONFIG.get("vllm", {})
VLLM_MAX_MODEL_LEN      = _vllm.get("max_model_len", 6144)
VLLM_MAX_NUM_SEQS       = _vllm.get("max_num_seqs", 64)
VLLM_MAX_BATCHED_TOKENS = _vllm.get("max_num_batched_tokens", 12288)
VLLM_GPU_MEM_UTIL       = _vllm.get("gpu_memory_utilization", 0.90)
VLLM_TP_SIZE            = _vllm.get("tensor_parallel_size", 1)   # default 1: avoids custom_all_reduce CUDA crash on T4 SM7.5
VLLM_ENFORCE_EAGER      = _vllm.get("enforce_eager", False)

print(f"Experiment : {EXPERIMENT}")
print(f"Model      : {MODEL_ID}")
print(f"max_model_len={VLLM_MAX_MODEL_LEN}  max_num_seqs={VLLM_MAX_NUM_SEQS}  max_tokens={MAX_TOKENS}  n={NUM_SAMPLES}  split={SPLIT}  tp={VLLM_TP_SIZE}")


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [ ]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [ ]:
# Load prompts from the experiment folder if available, otherwise use defaults.
_prompts_path = _find_exp_file(EXPERIMENT, "prompts.py")
if _prompts_path:
    _spec = importlib.util.spec_from_file_location("exp_prompts", _prompts_path)
    _mod  = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(_mod)
    SYSTEM_PROMPT_MATH = _mod.SYSTEM_PROMPT_MATH
    SYSTEM_PROMPT_MCQ  = _mod.SYSTEM_PROMPT_MCQ
    FEWSHOT_MATH       = getattr(_mod, "FEWSHOT_MATH", [])
    FEWSHOT_MCQ        = getattr(_mod, "FEWSHOT_MCQ",  [])
    print(f"Loaded prompts from {_prompts_path}")
else:
    print(f"No prompts.py for {EXPERIMENT!r} — using notebook defaults")
    SYSTEM_PROMPT_MATH = (
        "You are an expert mathematician. Solve the problem step-by-step. "
        "Put your final answer inside \\boxed{}. "
        "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
        "e.g. \\boxed{3, 7}."
    )
    SYSTEM_PROMPT_MCQ = (
        "You are an expert mathematician. "
        "Read the problem and the answer choices below, then select the single best answer. "
        "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
    )
    FEWSHOT_MATH: list = []
    FEWSHOT_MCQ:  list = []


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")


## 5. Load Model with vLLM

Key parameters are loaded from `experiments/<EXPERIMENT>/config.json`.
Defaults (if no config found):
- `tensor_parallel_size=2` — splits across both T4s
- `max_model_len=6144` — override in config.json `vllm.max_model_len`
- `dtype="float16"` — T4 (SM 7.5) doesn't support bfloat16
- `gpu_memory_utilization=0.90` — override in config.json `vllm.gpu_memory_utilization`


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    dtype="float16",                       # T4 doesn't support bfloat16
    tensor_parallel_size=VLLM_TP_SIZE,     # default 1: avoids custom_all_reduce crash on T4 SM7.5 + vLLM V1
    enforce_eager=VLLM_ENFORCE_EAGER,
    disable_custom_all_reduce=True,
    enable_prefix_caching=False,
    gpu_memory_utilization=VLLM_GPU_MEM_UTIL,
    max_model_len=VLLM_MAX_MODEL_LEN,
    trust_remote_code=True,
    max_num_seqs=VLLM_MAX_NUM_SEQS,
    max_num_batched_tokens=VLLM_MAX_BATCHED_TOKENS,
)

sampling_params = SamplingParams(
    n=NUM_SAMPLES,
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    min_p=0.0,
)

print("Model loaded.")


## 6. Generate Responses (public + private)

Run inference on both sets. Each response is **saved to JSONL immediately** so
that if any later cell crashes, you don't lose the multi-hour inference run.


In [ ]:
from collections import Counter

def extract_boxed(text):
    """Return the last \\boxed{...} value in text, or None."""
    matches = re.findall(r'\\boxed\{([^}]*)\}', text or "")
    return matches[-1].strip() if matches else None

def majority_vote(texts):
    """Return the text whose \\boxed{} answer appears most often. Falls back to texts[0]."""
    if len(texts) == 1:
        return texts[0]
    answers = [(extract_boxed(t), t) for t in texts]
    valid   = [(ans, t) for ans, t in answers if ans is not None]
    if not valid:
        return texts[0]
    modal = Counter(ans for ans, _ in valid).most_common(1)[0][0]
    return next(t for ans, t in valid if ans == modal)

def run_inference(items, out_path):
    """Generate responses with majority voting, save to JSONL, return voted responses."""
    prompts = []
    for item in items:
        system, user = build_prompt(item["question"], item.get("options"))
        fewshot = FEWSHOT_MCQ if item.get("options") else FEWSHOT_MATH
        messages = [{"role": "system", "content": system}, *fewshot, {"role": "user", "content": user}]
        prompts.append(tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        ))
    print(f"Generating {len(prompts)} responses (n={NUM_SAMPLES}) → {out_path} ...")
    outputs = llm.generate(prompts, sampling_params=sampling_params)

    responses = []
    for out in outputs:
        texts = [o.text.strip() for o in out.outputs]
        responses.append(majority_vote(texts))

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        for item, resp in zip(items, responses):
            f.write(json.dumps({
                "id": item["id"],
                "is_mcq": bool(item.get("options")),
                "response": resp,
            }) + "\n")
    print(f"Saved {len(responses)} responses to {out_path}")
    return responses


# Public / dev set
responses = run_inference(data, PUBLIC_RESP_PATH)

# Private set — skip on dev split to save runtime
if SPLIT != "dev":
    private_data = [json.loads(line) for line in open(PRIVATE_DATA_PATH)]
    print(f"Loaded {len(private_data)} private questions")
    private_responses = run_inference(private_data, PRIVATE_RESP_PATH)
else:
    print("Dev split — skipping private inference.")
    private_data, private_responses = [], []

# Preview
for i in range(min(2, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")


## 7. Build `submission.csv`

**Only private responses go in the submission** — the leaderboard test set is
`private.jsonl` (943 questions, IDs 0..942). `public.jsonl` has overlapping
IDs (0..1125), so mixing them together creates a dict collision and a broken
CSV. We include only private here.

Written **before** scoring so a scoring crash can never cost you the submission.


In [ ]:
import csv

if SPLIT == "dev":
    print("Dev split — no submission.csv generated (private inference was skipped).")
else:
    submission_rows = {item["id"]: resp for item, resp in zip(private_data, private_responses)}

    SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(SUBMISSION_PATH, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "response"])
        for qid in sorted(submission_rows.keys()):
            writer.writerow([qid, submission_rows[qid]])
    print(f"Saved {len(submission_rows)} responses to {SUBMISSION_PATH}")

    expected = len(private_data)
    assert len(submission_rows) == expected, f"Expected {expected} rows, got {len(submission_rows)}"
    print(f"OK — submission has {expected} rows matching private.jsonl.")


## 8. Score Public Responses (non-fatal)

Submission is already saved above, so this section is a bonus: local accuracy
feedback on the public set. Wrapped in try/except — if judger.py can't be
loaded or a question fails, the run still completes.


In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

results = []
judger = None
if JUDGER_DIR and JUDGER_DIR not in sys.path:
    sys.path.insert(0, JUDGER_DIR)
try:
    from judger import Judger
    judger = Judger(strict_extract=False)
    print(f"Judger loaded from {JUDGER_DIR}")
except Exception as e:
    print(f"Could not load judger ({e}). Skipping free-form scoring, "
          "doing MCQ-only local accuracy.")

for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]
    try:
        if is_mcq:
            correct = score_mcq(response, str(gold))
        elif judger is not None:
            gold_list = gold if isinstance(gold, list) else [gold]
            correct = judger.auto_judge(
                pred=response, gold=gold_list, options=[[]] * len(gold_list),
            )
        else:
            correct = None   # free-form, no judger → unknown
    except Exception:
        correct = False
    results.append({
        "id": item.get("id"), "is_mcq": is_mcq, "gold": gold,
        "response": response, "correct": correct,
    })

print(f"Scoring complete. {len(results)} results.")


## 9. Accuracy Summary

Counts only questions that were actually scored (skips free-form if no judger).


In [ ]:
mcq_res  = [r for r in results if r["is_mcq"] and r["correct"] is not None]
free_res = [r for r in results if not r["is_mcq"] and r["correct"] is not None]
scored   = mcq_res + free_res

def acc(subset):
    return sum(bool(r["correct"]) for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("LOCAL EVAL (public set)")
print("=" * 50)
print(f"  MCQ        : {sum(bool(r['correct']) for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(bool(r['correct']) for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(bool(r['correct']) for r in scored):4d} / {len(scored):4d}  ({acc(scored):.2f}%)")
print("=" * 50)

# Save scored results JSONL (for later analysis)
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(RESULTS_PATH, "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")
print(f"Saved {len(results)} records to {RESULTS_PATH}")
